In [4]:
import pandas as pd

# AES S-box
AES_SBOX = [
    0x63, 0x7C, 0x77, 0x7B, 0xF2, 0x6B, 0x6F, 0xC5, 0x30, 0x01, 0x67, 0x2B, 0xFE, 0xD7, 0xAB, 0x76,
    0xCA, 0x82, 0xC9, 0x7D, 0xFA, 0x59, 0x47, 0xF0, 0xAD, 0xD4, 0xA2, 0xAF, 0x9C, 0xA4, 0x72, 0xC0,
    0xB7, 0xFD, 0x93, 0x26, 0x36, 0x3F, 0xF7, 0xCC, 0x34, 0xA5, 0xE5, 0xF1, 0x71, 0xD8, 0x31, 0x15,
    0x04, 0xC7, 0x23, 0xC3, 0x18, 0x96, 0x05, 0x9A, 0x07, 0x12, 0x80, 0xE2, 0xEB, 0x27, 0xB2, 0x75,
    0x09, 0x83, 0x2C, 0x1A, 0x1B, 0x6E, 0x5A, 0xA0, 0x52, 0x3B, 0xD6, 0xB3, 0x29, 0xE3, 0x2F, 0x84,
    0x53, 0xD1, 0x00, 0xED, 0x20, 0xFC, 0xB1, 0x5B, 0x6A, 0xCB, 0xBE, 0x39, 0x4A, 0x4C, 0x58, 0xCF,
    0xD0, 0xEF, 0xAA, 0xFB, 0x43, 0x4D, 0x33, 0x85, 0x45, 0xF9, 0x02, 0x7F, 0x50, 0x3C, 0x9F, 0xA8,
    0x51, 0xA3, 0x40, 0x8F, 0x92, 0x9D, 0x38, 0xF5, 0xBC, 0xB6, 0xDA, 0x21, 0x10, 0xFF, 0xF3, 0xD2,
    0xCD, 0x0C, 0x13, 0xEC, 0x5F, 0x97, 0x44, 0x17, 0xC4, 0xA7, 0x7E, 0x3D, 0x64, 0x5D, 0x19, 0x73,
    0x60, 0x81, 0x4F, 0xDC, 0x22, 0x2A, 0x90, 0x88, 0x46, 0xEE, 0xB8, 0x14, 0xDE, 0x5E, 0x0B, 0xDB,
    0xE0, 0x32, 0x3A, 0x0A, 0x49, 0x06, 0x24, 0x5C, 0xC2, 0xD3, 0xAC, 0x62, 0x91, 0x95, 0xE4, 0x79,
    0xE7, 0xC8, 0x37, 0x6D, 0x8D, 0xD5, 0x4E, 0xA9, 0x6C, 0x56, 0xF4, 0xEA, 0x65, 0x7A, 0xAE, 0x08,
    0xBA, 0x78, 0x25, 0x2E, 0x1C, 0xA6, 0xB4, 0xC6, 0xE8, 0xDD, 0x74, 0x1F, 0x4B, 0xBD, 0x8B, 0x8A,
    0x70, 0x3E, 0xB5, 0x66, 0x48, 0x03, 0xF6, 0x0E, 0x61, 0x35, 0x57, 0xB9, 0x86, 0xC1, 0x1D, 0x9E,
    0xE1, 0xF8, 0x98, 0x11, 0x69, 0xD9, 0x8E, 0x94, 0x9B, 0x1E, 0x87, 0xE9, 0xCE, 0x55, 0x28, 0xDF,
    0x8C, 0xA1, 0x89, 0x0D, 0xBF, 0xE6, 0x42, 0x68, 0x41, 0x99, 0x2D, 0x0F, 0xB0, 0x54, 0xBB, 0x16,
]

def construct_ddt_values(sbox):
    """
    Construct the extended DDT where each entry contains the VALUES of a:

        T[Delta, delta] = { a | S(a ^ delta) ^ S(a) = Delta }

    with:
        Delta in {0, ..., 255}
        delta in {1, ..., 255}

    Returns:
        ddt_values[Delta][delta - 1] = list of integers a
    """
    n = len(sbox)
    ddt_values = [[[] for _ in range(1, n)] for _ in range(n)]

    for delta in range(1, n):      # delta = 1..255
        for a in range(n):         # a = 0..255
            Delta = sbox[a ^ delta] ^ sbox[a]
            ddt_values[Delta][delta - 1].append(a)

    return ddt_values

def to_hex_list(lst):
    return ", ".join(f"{x:02X}" for x in lst)

# Build the DDT with VALUES, not counts
ddt_values = construct_ddt_values(AES_SBOX)

# Make a display-friendly DataFrame like in your screenshot
ddt_display = pd.DataFrame(
    [[to_hex_list(cell) if cell else "" for cell in row] for row in ddt_values],
    index=[f"{Delta:02X}" for Delta in range(256)],
    columns=[str(delta) for delta in range(1, 256)]
)

# Optional: count table derived from the value table
ddt_counts = pd.DataFrame(
    [[len(cell) for cell in row] for row in ddt_values],
    index=[f"{Delta:02X}" for Delta in range(256)],
    columns=[str(delta) for delta in range(1, 256)]
)

print("Display table shape:", ddt_display.shape)   # (256, 255)
print("Count table shape:", ddt_counts.shape)      # (256, 255)
print("Differential uniformity:", ddt_counts.to_numpy().max())  # should be 4 for AES

# Example:
# This is the actual list of a values in T[0C, 1]
print("T[0C, 1] as integers:", ddt_values[0x0C][1 - 1])
print("T[0C, 1] as hex string:", to_hex_list(ddt_values[0x0C][1 - 1]))

# Show a small slice, similar to your example
ddt_display.iloc[:16, :9]

Display table shape: (256, 255)
Count table shape: (256, 255)
Differential uniformity: 4
T[0C, 1] as integers: [2, 3]
T[0C, 1] as hex string: 02, 03


,1,2,3,4,5,6,7,8,9
00,,,,,,,,,
01,"CE, CF",,,,,"30, 36","43, 44",,
02,,,"18, 1B",,,,,"F3, FB","84, 8D"
03,,"10, 12",,,,,,"30, 38","57, 5E"
04,"AC, AD","31, 33","05, 06",,"82, 87","59, 5F",,,
05,,"F0, F2","9C, 9F",,,"28, 2E",,,
06,"5C, 5D","4C, 4E",,,,,"1A, 1D",,"77, 7E"
07,,"01, 03",,,,,,,"B7, BE"
08,"94, 95",,"A1, A2",,"00, 05, 7A, 7F",,"50, 57",,
09,"24, 25",,,"B8, BC","18, 1D","91, 97","3B, 3C","80, 88",


In [8]:
import pandas as pd

def display_ddt_row(ddt_values, Delta, only_nonempty=False):
    """
    Display one whole row of the DDT for a given output difference Delta.

    Parameters
    ----------
    ddt_values : list
        The table returned by construct_ddt_values(sbox), where
        ddt_values[Delta][delta - 1] is the list of a values.
    Delta : int
        Output difference (0..255).
    only_nonempty : bool
        If True, only show columns where the entry is non-empty.

    Returns
    -------
    pandas.DataFrame
        A 1-row DataFrame suitable for display in Jupyter.
    """
    if not (0 <= Delta <= 255):
        raise ValueError("Delta must be between 0 and 255.")

    row = ddt_values[Delta]

    row_dict = {
        str(delta): ", ".join(f"{a:02X}" for a in row[delta - 1])
        for delta in range(1, 256)
    }

    if only_nonempty:
        row_dict = {k: v for k, v in row_dict.items() if v != ""}

    df_row = pd.DataFrame(row_dict, index=[f"{Delta:02X}"])
    return df_row

In [10]:
display_ddt_row(ddt_values, 0x69)

,1,2,3,4,5,6,7,8,9,10,...,246,247,248,249,250,251,252,253,254,255
69,"48, 49",,,,,"70, 76",,,,,...,"57, A1",,"55, AD","26, DF","41, BB","3F, C4",,"0B, F6","09, F7","69, 96"


In [11]:
display_ddt_row(ddt_values, 0x8c)

,1,2,3,4,5,6,7,8,9,10,...,246,247,248,249,250,251,252,253,254,255
8C,,"D9, DB",,,"42, 47",,,"E5, ED",,,...,"38, CE",,,,,"5A, A1",,"31, CC","54, AA","6C, 93"


In [12]:
display_ddt_row(ddt_values, 0x0f)

,1,2,3,4,5,6,7,8,9,10,...,246,247,248,249,250,251,252,253,254,255
0F,"74, 75","18, 1A",,,,,,"F4, FC",,"E3, E9",...,,"50, A7","6E, 96",,,,,"0A, F7",,"2E, D1"


In [13]:
import pandas as pd

def aes_xtime(x):
    """Multiply x by 0x02 in AES GF(2^8)."""
    x <<= 1
    if x & 0x100:
        x ^= 0x11B
    return x & 0xFF

def aes_mul(x, c):
    """Multiply x by a small constant c in AES GF(2^8)."""
    if c == 1:
        return x
    elif c == 2:
        return aes_xtime(x)
    elif c == 3:
        return aes_xtime(x) ^ x
    else:
        raise ValueError("This helper only supports multiplication by 1, 2, or 3.")

def find_good_deltas(ddt_values):
    """
    Find all delta in {1,...,255} such that the following four entries are all nonempty:
        [8C, 2*delta], [69, delta], [0C, delta], [0F, 3*delta]
    where 2*delta and 3*delta are AES GF(2^8) multiplications.

    Returns
    -------
    valid_deltas : list[int]
        List of delta values satisfying the condition.
    summary_df : pandas.DataFrame
        Jupyter-friendly table summarizing the matching deltas.
    """
    valid_deltas = []
    rows = []

    for delta in range(1, 256):
        delta2 = aes_mul(delta, 2)
        delta3 = aes_mul(delta, 3)

        e1 = ddt_values[0x8C][delta2 - 1]   # [8C, 2*delta]
        e2 = ddt_values[0x69][delta  - 1]   # [69, delta]
        e3 = ddt_values[0x0C][delta  - 1]   # [0C, delta]
        e4 = ddt_values[0x0F][delta3 - 1]   # [0F, 3*delta]

        if e1 and e2 and e3 and e4:
            valid_deltas.append(delta)
            rows.append({
                "delta": f"{delta:02X}",
                "2*delta": f"{delta2:02X}",
                "3*delta": f"{delta3:02X}",
                "|T[8C,2δ]|": len(e1),
                "|T[69,δ]|": len(e2),
                "|T[0C,δ]|": len(e3),
                "|T[0F,3δ]|": len(e4),
            })

    summary_df = pd.DataFrame(rows)
    return valid_deltas, summary_df

In [14]:
valid_deltas, summary_df = find_good_deltas(ddt_values)

print("Valid deltas:")
print([f"{d:02X}" for d in valid_deltas])

summary_df

Valid deltas:
['06', '28', '2E', '49', '58', '76', '8B', '90', 'A1', 'B2', 'B8', 'BC', 'D1', 'D3', 'E4']


,delta,2*delta,3*delta,"|T[8C,2δ]|","|T[69,δ]|","|T[0C,δ]|","|T[0F,3δ]|"
0,06,0C,0A,2,2,4,2
1,28,50,78,2,2,2,2
2,2E,5C,72,2,2,2,2
3,49,92,DB,2,2,2,2
4,58,B0,E8,2,2,2,2
5,76,EC,9A,2,2,2,2
6,8B,0D,86,2,2,2,2
7,90,3B,AB,2,2,2,2
8,A1,59,F8,2,2,2,2
9,B2,7F,CD,2,2,2,2
